# 🚀 Colab: Host 4-bit LoRA LLM via FastAPI (Unsloth Edition)

In [5]:

%%capture
!pip install unsloth vllm
!pip install git+https://github.com/huggingface/trl.git@e95f9fb74a3c3647b86f251b7e230ec51c64b72b
!pip install triton==3.1.0
!pip install -U pynvml
!pip install fastapi uvicorn pyngrok nest_asyncio -q
# !pip install numpy==1.23.5 --force-reinstall
!pip install -U numpy bitsandbytes transformers unsloth vllm triton==3.1.0 --force-reinstall

!pip install numpy>=1.25.0 --force-reinstall

In [6]:

from unsloth import FastLanguageModel, PatchFastRL, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/meta-Llama-3.1-8B-Instruct",  # You must have access to this model
    load_in_4bit=True,
    fast_inference=True,
    max_seq_length=512,
    max_lora_rank=16,
    gpu_memory_utilization=0.6,
)

model.load_lora("/content/drive/MyDrive/Misc College/LLMs/assn5_lora")


Unsloth: Patching Xformers to fix some performance issues.


<ipython-input-6-f0849c911f62>:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, PatchFastRL, is_bfloat16_supported


AttributeError: module 'numpy._core' has no attribute 'multiarray'

In [ ]:

from fastapi import FastAPI
from pydantic import BaseModel
from vllm import SamplingParams
import nest_asyncio
from pyngrok import ngrok
import uvicorn

app = FastAPI()

class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: int = 100
    temperature: float = 0.7

@app.post("/generate")
def generate(req: GenerateRequest):
    sampling = SamplingParams(
        temperature=req.temperature,
        max_tokens=req.max_tokens,
        top_p=0.95
    )
    output = model.fast_generate(req.prompt, sampling_params=sampling)[0].outputs[0].text
    return {"response": output}


In [ ]:

nest_asyncio.apply()
public_url = ngrok.connect(8000)
print(f"🚀 FastAPI endpoint: {public_url}/generate")

uvicorn.run(app, host="0.0.0.0", port=8000)
